# ЛР2. Пункт 9. Оценка адекватности модели

Выполняем две проверки по критерию Фишера при уровне значимости 0,05
по слайдам 16–18 лекции 3.

## Воспроизведение модели ЛР1

Из `all_data_processed.csv` берём 1365 строк с меткой `split=train`. Оцениваем МНК по этим строкам. Совпадение прогнозов проверяем по `predictions.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


data_dir = Path(".") if Path("predictions.csv").is_file() else Path("lab2")

data = pd.read_csv(data_dir / "all_data_processed.csv")
training = data.loc[data["split"].eq("train")]
feature_names = [
    "log1p_cores", "base_clock_ghz", "log1p_tdp", "log1p_l3",
    "age_centered", "manufacturer_Intel", "manufacturer_Other", "category_Laptop",
]
X = np.column_stack([np.ones(len(training)), training[feature_names]])
y = training["ln_cpu_mark"].to_numpy()
coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
y_hat = X @ coefficients

saved_predictions = pd.read_csv(data_dir / "predictions.csv")["y_hat"]
np.testing.assert_allclose(y_hat, saved_predictions, rtol=0, atol=1e-10)

## Проверка по коэффициенту детерминации

$H_0$ утверждает, что все восемь коэффициентов при факторах равны нулю.
Альтернатива состоит в том, что хотя бы один из них отличен от нуля.
При $k=8$ регрессорах и $n=1365$ наблюдениях
$F=\frac{R^2}{1-R^2}\frac{n-k-1}{k}$.

In [2]:
from scipy.stats import f

alpha = 0.05
n, p = X.shape
k = p - 1
df_res = n - p
rss = np.sum((y - y_hat) ** 2)
tss = np.sum((y - y.mean()) ** 2)
r2 = 1 - rss / tss

f_r2 = (r2 / (1 - r2)) * (df_res / k)
critical_r2 = f.isf(alpha, k, df_res)
p_r2 = f.sf(f_r2, k, df_res)

print(f"R² = {r2:.6f}; степени свободы F: ({k}, {df_res})")
p_r2_text = f"{p_r2:.3g}" if p_r2 > 0 else "< 1e-300"
print(f"Fрасч = {f_r2:.4f}; Fкрит = {critical_r2:.4f}; p-value {p_r2_text}")
if f_r2 > critical_r2:
    print("H₀ отклоняется. Факторы совместно улучшают модель со свободным членом.")
else:
    print("Оснований отклонить H₀ по общему F-тесту недостаточно.")

R² = 0.884312; степени свободы F: (8, 1356)
Fрасч = 1295.6426; Fкрит = 1.9452; p-value < 1e-300
H₀ отклоняется. Факторы совместно улучшают модель со свободным членом.


## Проверка по остаточной дисперсии

Проверяем равенство дисперсий по верхнему критическому значению, как в
алгоритме на слайде 18. Большие значения
$F=S_{\mathrm{ост}}^2/S_0^2$ свидетельствуют об избыточном разбросе остатков.
Остаточная дисперсия $S_{\mathrm{ост}}^2=\mathrm{RSS}/(n-p)$
вычисляется на обучающей выборке. $S_0^2$ берём из согласованной группы похожих CPU.

In [3]:
extra_sample = pd.read_csv(data_dir / "cpu_extra_sample.csv")
y_extra = np.log(extra_sample["cpu_mark"].to_numpy())
df_0 = len(y_extra) - 1
S0_sq = y_extra.var(ddof=1)

# Сопоставляем записи дополнительной группы по всем её сохранённым полям.
shared_columns = extra_sample.columns.tolist()
development = training[["raw_" + column for column in shared_columns]].set_axis(shared_columns, axis=1)
extra_keys = pd.MultiIndex.from_frame(extra_sample[shared_columns])
development_keys = pd.MultiIndex.from_frame(development[shared_columns])
overlap = extra_keys.isin(development_keys).sum()

print(f"Дополнительная группа: {len(extra_sample)} CPU, ν₀ = {df_0}")
print(f"S₀² = {S0_sq:.6f}")
print(f"Совпадений с обучающей выборкой: {overlap} из {len(extra_sample)}")

Дополнительная группа: 20 CPU, ν₀ = 19
S₀² = 0.080688
Совпадений с обучающей выборкой: 14 из 20


In [4]:
S_ost_sq = rss / df_res
f_res = S_ost_sq / S0_sq
critical_res = f.isf(alpha, df_res, df_0)
p_res = f.sf(f_res, df_res, df_0)

print(f"Sост² = {S_ost_sq:.6f}; степени свободы F: ({df_res}, {df_0})")
print(f"Fрасч = {f_res:.4f}; Fкрит = {critical_res:.4f}; приближённое p-value = {p_res:.4f}")
if f_res > critical_res:
    print("По критерию лекции остаточная дисперсия превышает допустимый уровень.")
else:
    print("По критерию лекции оснований считать остаточную дисперсию избыточной недостаточно.")

Sост² = 0.164369; степени свободы F: (1356, 19)
Fрасч = 2.0371; Fкрит = 1.8828; приближённое p-value = 0.0329
По критерию лекции остаточная дисперсия превышает допустимый уровень.


Группа состоит из разных CPU с близкими характеристиками, поэтому её дисперсия
включает различия между процессорами. Пересечение с обучающими данными также
ставит под сомнение предпосылку независимости двух оценок дисперсии.
Результат второй проверки следует трактовать как приближённый.